In [1]:
import os
import torch
import timm
import numpy as np
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from timm.data.constants import IMAGENET_DEFAULT_MEAN, IMAGENET_DEFAULT_STD
from tqdm import tqdm

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 50


In [2]:
DATA_ROOT = r"D:\Project\archive"

train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(0.1, 0.1, 0.1, 0.05),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_DEFAULT_MEAN, IMAGENET_DEFAULT_STD),
])

eval_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_DEFAULT_MEAN, IMAGENET_DEFAULT_STD),
])

train_ds = datasets.ImageFolder(os.path.join(DATA_ROOT, "train"), transform=train_tf)
val_ds   = datasets.ImageFolder(os.path.join(DATA_ROOT, "val"), transform=eval_tf)
test_ds  = datasets.ImageFolder(os.path.join(DATA_ROOT, "test"), transform=eval_tf)

NUM_CLASSES = len(train_ds.classes)
print("Classes:", train_ds.classes)

train_loader = DataLoader(train_ds, BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True)
val_loader   = DataLoader(val_ds, BATCH_SIZE*2, shuffle=False, num_workers=4, pin_memory=True)
test_loader  = DataLoader(test_ds, BATCH_SIZE*2, shuffle=False, num_workers=4, pin_memory=True)


Classes: ['0_normal', '1_ulcerative_colitis', '2_polyps', '3_esophagitis']


In [3]:
class EFFResNetViT(nn.Module):
    def __init__(self, num_classes):
        super().__init__()

        # EfficientNet-B4
        self.eff = timm.create_model(
            "efficientnet_b4",
            pretrained=True,
            features_only=True
        )
        eff_dim = self.eff.feature_info[-1]['num_chs']

        # ResNet-50
        self.res = timm.create_model(
            "resnet50",
            pretrained=True,
            features_only=True
        )
        res_dim = self.res.feature_info[-1]['num_chs']

        fused_dim = eff_dim + res_dim

        # Fusion projection
        self.fusion = nn.Conv2d(fused_dim, 768, kernel_size=1)

        # Transformer Encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=768,
            nhead=12,
            dim_feedforward=3072,
            dropout=0.2,
            batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=3)

        # Regularized Classifier
        self.classifier = nn.Sequential(
            nn.LayerNorm(768),
            nn.Dropout(0.4),
            nn.Linear(768, num_classes)
        )

    def forward(self, x):
        eff_feat = self.eff(x)[-1]   # [B, Ce, H, W]
        res_feat = self.res(x)[-1]   # [B, Cr, H, W]

        fused = torch.cat([eff_feat, res_feat], dim=1)
        tokens = self.fusion(fused)
        tokens = tokens.flatten(2).transpose(1, 2)
        tokens = self.transformer(tokens)
        pooled = tokens.mean(dim=1)

        return self.classifier(pooled)


In [4]:
model = EFFResNetViT(NUM_CLASSES).to(DEVICE)

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

optimizer = optim.AdamW(
    model.parameters(),
    lr=3e-4,
    weight_decay=0.05
)

scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer,
    T_0=10,
    T_mult=2,
    eta_min=1e-6
)


Unexpected keys (bn2.num_batches_tracked, bn2.bias, bn2.running_mean, bn2.running_var, bn2.weight, classifier.bias, classifier.weight, conv_head.weight) found while loading pretrained weights. This may be expected if model is being adapted.


In [5]:
class EarlyStopping:
    def __init__(self, patience=7):
        self.patience = patience
        self.best_acc = 0.0
        self.counter = 0

    def step(self, acc, model):
        if acc > self.best_acc:
            self.best_acc = acc
            self.counter = 0
            torch.save(model.state_dict(), "best_effresnetvit.pth")
            return False
        else:
            self.counter += 1
            return self.counter >= self.patience


In [6]:
early_stopper = EarlyStopping(patience=7)

for epoch in range(EPOCHS):
    model.train()
    train_loss = 0.0

    for imgs, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}"):
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

    scheduler.step()

    model.eval()
    correct = 0
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            preds = model(imgs).argmax(1)
            correct += (preds == labels).sum().item()

    val_acc = correct / len(val_loader.dataset)
    print(f"Epoch {epoch+1} | Val Acc: {val_acc:.4f}")

    if early_stopper.step(val_acc, model):
        print("🛑 Early stopping triggered")
        break


Epoch 1/50: 100%|██████████| 100/100 [00:28<00:00,  3.53it/s]


Epoch 1 | Val Acc: 0.8745


Epoch 2/50: 100%|██████████| 100/100 [00:27<00:00,  3.69it/s]


Epoch 2 | Val Acc: 0.9365


Epoch 3/50: 100%|██████████| 100/100 [00:27<00:00,  3.65it/s]


...

Epoch 8 | Val Acc: 0.8580


Epoch 9/50: 100%|██████████| 100/100 [00:29<00:00,  3.44it/s]


Epoch 9 | Val Acc: 0.9015


Epoch 10/50: 100%|██████████| 100/100 [00:28<00:00,  3.49it/s]


Epoch 10 | Val Acc: 0.8945
🛑 Early stopping triggered


In [7]:
model.load_state_dict(torch.load("best_effresnetvit.pth"))
model.eval()

correct = 0
with torch.no_grad():
    for imgs, labels in test_loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        preds = model(imgs).argmax(1)
        correct += (preds == labels).sum().item()

print("Test Accuracy:", correct / len(test_loader.dataset))


Test Accuracy: 0.955
